# Debug notebook — model + reward functions sanity check
Load a sample, run the model, run the reward functions, see everything.

In [22]:
# ── Config ────────────────────────────────────────────────────────────────────
MODEL_PATH  = "/home/rlvr_experiments/outputs/sft_qwen_0.5b"   # base or SFT checkpoint
TASK_NAME   = "countdown"                   # any reasoning-gym task
TASK_PARAMS = {"min_numbers": 2,"max_numbers": 3}                            # e.g. {"min_numbers": 4}
SEED        = 42
SAMPLE_IDX  = 0                             # which sample to inspect

MAX_NEW_TOKENS = 512
TEMPERATURE    = 0.7

In [23]:
import sys
sys.path.insert(0, "..")

import torch
import reasoning_gym
from transformers import AutoModelForCausalLM, AutoTokenizer
from environments import load_environment

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}")

device: cuda


In [24]:
# ── Build environment (gives us reward functions + dataset) ───────────────────
config = {
    "environment": {
        "name": "reasoning_gym",
        "task_name": TASK_NAME,
        "samples": 50,
        "task_params": TASK_PARAMS,
    },
    "seed": SEED,
}

env = load_environment(config)
dataset = env.get_dataset(config)
reward_fns = env.get_reward_functions()

print(f"\nDataset size: {len(dataset)}")
print(f"Columns: {dataset.column_names}")
print(f"Reward functions: {[fn.__name__ for fn in reward_fns]}")

Generating 50 samples for 'countdown'...
  Task params: {'min_numbers': 2, 'max_numbers': 3}
  Generated 50 samples
  Sample answer: 4*95

Dataset size: 50
Columns: ['prompt', 'ground_truth', 'task', 'metadata']
Reward functions: ['correctness_reward', 'format_reward']


In [25]:
# ── Inspect a sample ──────────────────────────────────────────────────────────
sample = dataset[SAMPLE_IDX]
print("prompt (messages):")
for msg in sample['prompt']:
    print(f"  [{msg['role']}] {msg['content']}")
print(f"\nground_truth: {sample['ground_truth']}")
print(f"metadata:     {sample.get('metadata', {})}")

prompt (messages):
  [user] Using all the numbers 95, 4, create an expression that equals 380.
You can only use each number once.

Final answer format instructions:
1. Provide your solution as a arithmetic expression (no '=' sign).
2. Do not include the target number in the expression.
3. Use '*' for multiplication.
4. Use '/' for division.
5. Do not include any other text or formatting.


ground_truth: 4*95
metadata:     {'difficulty': {'numbers': [2, 3], 'target': [100, 999], 'value': [1, 100]}, 'expression': '4*95', 'numbers': [95, 4], 'source_dataset': 'countdown', 'source_index': 0, 'target': 380}


In [5]:
# ── Load model ────────────────────────────────────────────────────────────────
print(f"Loading: {MODEL_PATH}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    device_map="auto",
)
model.eval()
print("done")

Loading: /home/rlvr_experiments/outputs/sft_qwen_0.5b
done


In [26]:
# ── Run model ─────────────────────────────────────────────────────────────────
system_prompt = env.get_system_prompt() if hasattr(env, 'get_system_prompt') else (
    "You are a helpful assistant. You must always think step-by-step. Put your thinking process inside <think> and </think> tags, and put your final answer inside <answer> and </answer> tags."
)

# build messages: add system prompt + the user turn from the dataset
messages = [{"role": "system", "content": system_prompt}] + sample['prompt']

prompt_text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
with torch.no_grad():
    out_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        do_sample=TEMPERATURE > 0,
        pad_token_id=tokenizer.pad_token_id,
    )

new_tokens = out_ids[0][inputs["input_ids"].shape[1]:]
response = tokenizer.decode(new_tokens, skip_special_tokens=False)

print("=" * 60)
print("RAW MODEL OUTPUT")
print("=" * 60)
print(response)

RAW MODEL OUTPUT
<think>
I have the numbers 95, 4, and I need to form an expression that equals 380 using each number exactly once.
Let's start by considering possible combinations. One approach is to look for factors of 380 that could be combined with 4 and 95.
95 * 4 = 380
This works because 95 * 4 = 380.
The remaining factor is 495, but it cannot be used since we only have four digits.
Therefore, my expression is 95 * 4 = 380.
</think>
<answer>
95 * 4 = 380
</answer><|im_end|>


In [13]:
print(prompt_text)

<|im_start|>system
You are a helpful assistant. You must always think step-by-step. Put your thinking process inside <think> and </think> tags, and put your final answer inside <answer> and </answer> tags.<|im_end|>
<|im_start|>user
Calculate 139 using all of these numbers: 36, 29, 95, 32, 4, 15.
Each number may be used at most once.

Final answer format instructions:
1. Provide your solution as a arithmetic expression (no '=' sign).
2. Do not include the target number in the expression.
3. Use '*' for multiplication.
4. Use '/' for division.
5. Do not include any other text or formatting.
<|im_end|>
<|im_start|>assistant



In [ ]:
# ── Shared pre-processing (used by both reward fns) ───────────────────────────
import re

# Match the shape TRL passes in
completions  = [[{"role": "assistant", "content": response}]]
ground_truth = [sample['ground_truth']]
metadata     = [sample.get('metadata', {})]

# Step 1: unwrap completion (env._unwrap_completion)
unwrapped = env._unwrap_completion(completions[0])
print("── unwrapped completion ──")
print(repr(unwrapped))

# Step 2: extract answer (env._extract_answer)
extracted = env._extract_answer(unwrapped)
print(f"\n── extracted answer ──")
print(repr(extracted))

In [ ]:
# ── format_reward breakdown ───────────────────────────────────────────────────
has_think_open  = '<think>'  in unwrapped
has_think_close = '</think>' in unwrapped
has_ans_open    = '<answer>' in unwrapped
has_ans_close   = '</answer>' in unwrapped

print(f"<think>    present: {has_think_open}")
print(f"</think>   present: {has_think_close}")
print(f"<answer>   present: {has_ans_open}")
print(f"</answer>  present: {has_ans_close}")

score = 0.0
if has_think_open and has_think_close:
    score += 0.3;  print("\nthink pair complete  → +0.3")
elif has_think_open:
    score += 0.1;  print("\nthink open only      → +0.1")

if has_ans_open and has_ans_close:
    score += 0.3;  print("answer pair complete → +0.3")
elif has_ans_open:
    score += 0.1;  print("answer open only     → +0.1")

if has_think_open and has_think_close and has_ans_open and has_ans_close:
    score = 1.0;   print("both complete → bonus, score set to 1.0")

print(f"\nformat_reward  →  {score:.3f}")
print(f"env call check →  {env.format_reward(completions)[0]:.3f}")

In [ ]:
# ── correctness_reward breakdown ─────────────────────────────────────────────
truth    = ground_truth[0]
meta     = metadata[0]
task     = TASK_NAME

print(f"extracted : {repr(extracted)}")
print(f"truth     : {repr(truth)}")
print(f"task      : {task}")
print(f"metadata  : {meta}")
print()

if task == 'countdown':
    target = meta.get('target') if isinstance(meta, dict) else None
    print(f"target from metadata: {target}")

    cleaned = (extracted or '').strip()
    safe    = bool(re.match(r'^[\d\s\+\-\*\/\(\)\.]+$', cleaned))
    print(f"expression safe (digits+ops only): {safe}")

    if safe:
        try:
            result = eval(cleaned)
            print(f"eval({repr(cleaned)}) = {result}")
            print(f"result == target: {result == target}")
        except Exception as e:
            print(f"eval failed: {e}")
    else:
        print("expression contains non-safe chars — skipping eval")
else:
    ext_norm   = env._normalize_answer(extracted or '')
    truth_norm = env._normalize_answer(truth)
    print(f"normalized extracted : {repr(ext_norm)}")
    print(f"normalized truth     : {repr(truth_norm)}")
    print(f"exact match          : {ext_norm == truth_norm}")

print()
print(f"correctness_reward →  {env.correctness_reward(completions, ground_truth=ground_truth, metadata=metadata)[0]:.3f}")